# K1 - Train a custom baseline on Kaggle (free GPU, ~30 h/week per person)

**Notebook settings (right panel):** Accelerator -> *GPU T4 x2* (only one is used; same chip as Colab, so
timings compare). Internet -> *On* (needs a phone-verified Kaggle account). **Add Input** -> the team's
`umd-prepared` dataset. Private repo? Add-ons -> Secrets -> `GITHUB_TOKEN`.

**Always run with Save Version -> *Save & Run All (Commit)*.** It runs in the background (up to 12 h) and
keeps `/kaggle/working` as the version's output. An interactive session loses everything when it stops.

**To resume** after a 12 h run: Add Input -> *Notebook Output Files* -> this notebook's latest version,
then Save & Run All again. The cell below picks up the newest `last.pt` automatically.

In [ ]:
MODEL = 'unet3d'       # 'unet3d' | 'aksunet' (or any custom model)
SMOKE = True           # True: 3-epoch timing run. False: the real run
EPOCHS = 200           # must equal the Colab custom models' EPOCHS
MAX_HOURS = 11.0       # Kaggle stops sessions at 12 h; leave time to save outputs
PATCH = '192 192 16'
BATCH = 2

In [ ]:
import glob, os, shutil, subprocess, zipfile
import torch

REPO_URL = 'https://github.com/DeekshaBandi/test.git'
try:
    from kaggle_secrets import UserSecretsClient
    REPO_URL = REPO_URL.replace('https://', f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo
REPO = '/tmp/figo-repo'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
%cd {REPO}
!pip -q install -r requirements.txt

def locate(name):
    """Find a file/folder from the input dataset; unzip it if Kaggle kept it as a .zip."""
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if hits:
        return hits[0]
    zips = glob.glob(f'/kaggle/input/**/{name}.zip', recursive=True)
    if not zips:
        raise FileNotFoundError(f'{name} not found - did you add the umd-prepared dataset as input?')
    dst = f'/tmp/data/{name}'
    if not os.path.exists(dst):
        zipfile.ZipFile(zips[0]).extractall(dst)
    inner = glob.glob(f'{dst}/**/{name}', recursive=True)   # zip may contain the folder itself
    return inner[0] if inner and os.path.isdir(inner[0]) else dst

SPLITS, CACHE, CLEAN = locate('splits.json'), locate('cache_0.9mm'), locate('clean')
print(SPLITS, CACHE, CLEAN, len(os.listdir(CACHE)), 'cached volumes')

OUT = '/kaggle/working'
RUN = f'{OUT}/runs/' + (f'smoke_{MODEL}' if SMOKE else MODEL)
if SMOKE:
    EPOCHS = 3
# resume: copy the most advanced previous run attached as input
prev = glob.glob(f'/kaggle/input/**/runs/{os.path.basename(RUN)}/last.pt', recursive=True)
if prev and not os.path.exists(f'{RUN}/last.pt'):
    best = max(prev, key=lambda p: torch.load(p, map_location='cpu', weights_only=False)['epoch'])
    shutil.copytree(os.path.dirname(best), RUN, dirs_exist_ok=True)
    print('resuming from', best)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m figo.train --model {MODEL} --cache-dir {CACHE} --splits {SPLITS} --out-dir {RUN} \
    --epochs {EPOCHS} --max-hours {MAX_HOURS} --patch-size {PATCH} --batch-size {BATCH} --workers 3

In [ ]:
# timing -> how many epochs fit the budget (row 1 = pure training, row 2 = training + validation)
import csv
log = list(csv.DictReader(open(f'{RUN}/log.csv')))
for r in log:
    print('epoch', r['epoch'], r['epoch_sec'], 's')
if len(log) >= 3:
    t_train = float(log[1]['epoch_sec'])
    t_val = max(float(log[2]['epoch_sec']) - t_train, 0)
    per_epoch = t_train + t_val / 10          # validation runs every 10 epochs
    print(f'~{per_epoch:.0f} s per epoch on average; validation ~{t_val:.0f} s')
    for h in (6, 8, 10):
        print(f'  {h:2d} GPU-hours  ->  EPOCHS = {int(h * 3600 / per_epoch)}')
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## Test-set inference and evaluation (in the run where training finishes)

In [ ]:
import json
done = torch.load(f'{RUN}/last.pt', map_location='cpu', weights_only=False)['epoch'] + 1 >= EPOCHS
if done and not SMOKE:
    PRED, RES = f'{OUT}/preds/{MODEL}', f'{OUT}/results'
    !python -m figo.infer --checkpoint {RUN}/best.pt --data-dir {CLEAN} --splits {SPLITS} --subset test --out-dir {PRED}
    !python -m figo.evaluate --pred-dir {PRED} --data-dir {CLEAN} --splits {SPLITS} --subset test --out {RES}/{MODEL}_seg.csv
    !python -m figo.figo_eval agreement --ref-dir {CLEAN} --pred-dir {PRED} --splits {SPLITS} --subset test \
        --out {RES}/{MODEL}_figo_agreement.csv
    print("Download this version's output and copy preds/ and results/ into MyDrive/figo/ for model selection.")
else:
    print('Training not finished yet - attach this output as input and Save & Run All again.')